# Load neuron mesh and spines with head/neck classification

This notebook demonstrates how to:
1. Load a neuron mesh from an `.obj` file and scale it by 1e-3 (nm → µm)
2. Load the corresponding `.h5` file containing spines with head/neck data
3. Access head-only and neck-only spine meshes

In [2]:
import trimesh
from trimesh import util as triutil
from trimesh.visual.color import ColorVisuals

from morph_spines import load_morphology_with_spines

## Load and scale the neuron mesh

The `.obj` mesh is in nanometers; scale by 1e-3 to convert to micrometers.

In [ ]:
mesh_path = "/scratch/spine_extraction/123456.obj"
neuron_mesh = trimesh.load(mesh_path)
neuron_mesh.vertices *= 1e-3

print(f"Neuron mesh: {len(neuron_mesh.vertices)} vertices, {len(neuron_mesh.faces)} faces")

## Load the morphology with spines (head/neck data)

Load the `.h5` file with `load_meshes=True` to preload spine meshes including
head/neck triangle classification.

In [ ]:
h5_path = "/scratch/spine_extraction/123456_spine_head_neck/123456.h5"
m = load_morphology_with_spines(h5_path, spines_are_centered=True, load_meshes=True)

print(f"Spine count: {m.spines.spine_count}")

## Access spine head and neck meshes

Use `include_head=False` to get only the neck, or `include_neck=False` to get only the head.

In [ ]:
spine_idx = 0

full_mesh = m.spines.spine_mesh(spine_idx)
neck_mesh = m.spines.spine_mesh(spine_idx, include_head=False)
head_mesh = m.spines.spine_mesh(spine_idx, include_neck=False)

print(f"Full spine: {len(full_mesh.faces)} faces")
print(f"Neck only:  {len(neck_mesh.faces)} faces")
print(f"Head only:  {len(head_mesh.faces)} faces")

## Visualize head and neck with different colors

In [ ]:
neck_mesh.visual = ColorVisuals(mesh=neck_mesh, face_colors=[100, 200, 100])
head_mesh.visual = ColorVisuals(mesh=head_mesh, face_colors=[255, 100, 100])

combined = triutil.concatenate([neck_mesh, head_mesh])
combined.show()

## Overlay spines on the neuron mesh

In [ ]:
# neuron_mesh.visual = ColorVisuals(mesh=neuron_mesh, face_colors=[200, 200, 200, 10])

import numpy as np

neuron_mesh.visual = ColorVisuals(
    mesh=neuron_mesh,
    face_colors=np.full((len(neuron_mesh.faces), 4), [255, 255, 200, 10], dtype=np.uint8)
)

# Collect all spine head/neck meshes with colors
spine_parts = []
for i in range(min(m.spines.spine_count, 250)):  # limit for performance
    neck = m.spines.spine_mesh(i, include_head=False)
    head = m.spines.spine_mesh(i, include_neck=False)
    neck.visual = ColorVisuals(mesh=neck, face_colors=[100, 200, 100])
    head.visual = ColorVisuals(mesh=head, face_colors=[255, 100, 100])
    spine_parts.extend([neck, head])

scene = triutil.concatenate(spine_parts)
scene.show()